In [19]:
#!/Users/ginoprasad/miniconda3/envs/google-api/bin/python3

import os, glob
import sys
import settings
import json
import concurrent.futures
from google.cloud import pubsub_v1
import google.auth
import subprocess as sp
import time
import base64
import re

import utils
from tqdm import tqdm

In [2]:
def extract_text(payload):
    if 'parts' in payload:
        return ''.join(map(extract_text, payload['parts']))
    elif payload.get('mimeType') == 'text/plain':
        data = payload.get('body', {}).get('data', '')
        if data:
            return base64.urlsafe_b64decode(data).decode('utf-8', errors='ignore')
    return ''


list_res = utils.service.users().messages().list(userId='me', q='in:inbox', maxResults=1000).execute()
messages = list_res.get('messages', [])

In [43]:
regex_include = [
    r'(?<=Get Code\r\n\[)https://.*?(?=[ \]])',
    r'(?<=Enter this code to sign in\r\n\r\n)[0-9][0-9][0-9][0-9]'
]

regex_exclude = [
    'Please review who’s using your Netflix account',
    'We’ve updated your account with your new payment info'
]

emails = [
    'info@account.netflix.com'
]

In [42]:
for message in tqdm(messages):
    eid = message['id']
    msg = utils.service.users().messages().get(
        userId='me', id=eid, format='full'
    ).execute()

    
    email_from = ''.join([x['value'] for x in msg['payload']['headers'] if x['name'] == 'From'])
    email_from = re.search(r'(?<=<).*(?=>)', email_from).group()
    
    labels = msg.get('labelIds', [])
    print(email_from)
    if 'INBOX' in labels and 'SENT' not in labels and email_from in emails:
        payload = extract_text(msg['payload'])

        exclude = False
        for regex in regex_exclude:
            code = re.search(regex, payload)
            if code is not None:
                exclude = True
                break
        if exclude:
            print("HERE")
            continue
                
        for regex in regex_include:
            code = re.search(regex, payload)
            if code is not None:
                code = code.group()
                break
        
        if code is None:
            with open(settings.payload_path, 'w') as outfile:
                outfile.write(payload)


            print(payload)
            break
        else:
            print(code)

  0%|                          | 1/500 [00:00<01:58,  4.20it/s]

info@account.netflix.com
HERE


  0%|                          | 2/500 [00:00<02:09,  3.85it/s]

info@account.netflix.com
1073


  1%|▏                         | 3/500 [00:00<02:00,  4.11it/s]

...

no-reply@indeed.com


  5%|█▏                       | 24/500 [00:05<01:58,  4.03it/s]

noreply@glassdoor.com


  5%|█▏                       | 24/500 [00:06<02:01,  3.91it/s]

info@account.netflix.com
We’ve updated your account with your new payment info, as you asked. Your membership will automatically continue as long as you choose to remain a member.

Payment method updated

Hi Neil,

We’ve updated your account with your new payment info, as
you asked. Your membership will automatically continue as
long as you choose to remain a member.

 

Payment

   •••• •••••• • 1009

 

We're here to help if you need it. Visit the Help Center
[https://help.netflix.com/support/244?g=f5053bdc-afd3-46db-818d-d2caf2f29cb5&lkid=URL_HELP&lnktrk=EVO]
for more info or contact us
[https://help.netflix.com/contactus?g=f5053bdc-afd3-46db-818d-d2caf2f29cb5&lkid=URL_CONTACT&lnktrk=EVO].

The Netflix team

 

View All TV Shows & Movies
[https://www.netflix.com/browse?g=f5053bdc-afd3-46db-818d-d2caf2f29cb5&lkid=URL_ESCAPE_HATCH&lnktrk=EVO]
 

   Questions? Call 1-844-569-7700
   
   121 Albright Way, Los Gatos, CA 95032, U.S.A.
   [https://help.netflix.com/legal/corpinfo?g=f5053bdc

In [26]:
re.search(r'(?<=<).*(?=>)', email_from).group()

'ginoprasad@gmail.com'

In [37]:
msg['snippet']

'Please review who&#39;s using your Netflix account, ginoprasad3@gmail.com. ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏'

In [34]:
code

In [32]:
for regex in regex_list:
    code = re.search(regex, payload)
    if code is not None:
        code = code.group()
        break
print(

None
None


In [15]:





# labels = msg.get('labelIds', [])
# if 'INBOX' in labels and 'SENT' not in labels and 'info@account.netflix.com' in email_from:
#     payload = extract_text(msg['payload'])
#     with open("/Users/ginoprasad/Scripts/EmailManager/test.txt", 'w') as outfile:
#         outfile.write(payload)





        
    # link = re.search(r'(?<=Get Code\r\n\[)https://.*?(?=[ \]])', payload).group()
    # message = f'Netflix Code: {link}'
    # assert len(set("\n'\"") - set(message)) == 3

    # cmd = f"'{settings.send_text_path}' '{settings.group_chat_id}' '{message}'"
    # sp.run(cmd, shell=True)

In [18]:
code

'6646'

In [14]:

code

In [10]:
payload

'Enter this code to sign in\r\n\r\nEnter this code to sign in\r\n\r\n6646\r\n\r\nEnter the code above on your device to sign in to Netflix.\r\nThis code will expire in 15 minutes.\r\n\r\nIf you didn’t send this request, you can ignore this email\r\nor review your recent device activity.\r\n[https://www.netflix.com/accountaccess?g=5c03f486-1f81-4951-bb10-eb7d59d9bac1&lkid=URL_ACCOUNT_ACCESS&lnktrk=EVO]\r\n\r\nTo help security, don’t share this code with anyone outside\r\nyour household.\r\n\r\nThe Netflix team\r\n\r\n\xa0\r\n\xa0\r\n\r\n   Questions? Call 1-844-569-7700\r\n   \r\n   121 Albright Way, Los Gatos, CA 95032, U.S.A.\r\n   [https://help.netflix.com/legal/corpinfo?g=5c03f486-1f81-4951-bb10-eb7d59d9bac1&lkid=URL_CORP_INFO&lnktrk=EVO]\r\n   \r\n   Terms of Use\r\n   [https://www.netflix.com/TermsOfUse?g=5c03f486-1f81-4951-bb10-eb7d59d9bac1&lkid=URL_TERMS&lnktrk=EVO]\r\n   Privacy\r\n   [https://www.netflix.com/PrivacyPolicy?g=5c03f486-1f81-4951-bb10-eb7d59d9bac1&lkid=URL_PRIVACY

In [ ]:
link = .group()

In [ ]:


6646